<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name: Juifan Lin
<br>
Github Username: JiaLiiin
<br>
USC ID: 9985422205

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [4]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### (b) Test and Train Data

In [5]:
data_dir = os.path.join('..', 'Data', 'AReM')
folders = ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']

train_files = []
test_files = []

for folder in folders:
    folder_path = os.path.join(data_dir, folder)
    if not os.path.exists(folder_path):
        continue
        
    for file in os.listdir(folder_path):
        if not file.endswith('.csv'): 
            continue
        file_path = os.path.join(folder_path, file)
        if folder in ['bending1', 'bending2']:
            if file in ['dataset1.csv', 'dataset2.csv']:
                test_files.append((folder, file_path))
            else:
                train_files.append((folder, file_path))
        else:
            if file in ['dataset1.csv', 'dataset2.csv', 'dataset3.csv']:
                test_files.append((folder, file_path))
            else:
                train_files.append((folder, file_path))

print(f"Training instances: {len(train_files)}")
print(f"Testing instances: {len(test_files)}")

Training instances: 69
Testing instances: 19


### (c) Feature Extraction

#### i. Research

Central Tendency: Mean, Median.   
Dispersion: Standard Deviation, Variance, Interquartile Range, Range (Max-Min).
Extremes: Minimum, Maximum.   
Shape: Skewness, Kurtosis.
Quartiles: 1st quartile, 3rd quartile.

#### ii. Extraction

In [6]:
columns_of_interest = ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'vg_rss23', 'ar_rss23']
metrics = ['min', 'max', 'mean', 'median', 'std', '1st_quart', '3rd_quart']

def extract_features(file_list):
    dataset_features = []
    
    for folder, path in file_list:
        try:
            df = pd.read_csv(path, skiprows=4)
            if df.shape[1] == 1: 
                df = pd.read_csv(path, skiprows=4, sep='\s+')
            df = df.iloc[:, 1:7]
            df.columns = columns_of_interest
            df = df.apply(pd.to_numeric, errors='coerce').dropna()
            
            row_features = []
            for col in columns_of_interest:
                row_features.extend([
                    df[col].min(),
                    df[col].max(),
                    df[col].mean(),
                    df[col].median(),
                    df[col].std(),
                    df[col].quantile(0.25),
                    df[col].quantile(0.75)
                ])
            row_features.append(folder)
            dataset_features.append(row_features)
        except Exception as e:
            print(f"Error processing {path}: {e}")

    feature_cols = []
    for i in range(1, 7):
        for metric in metrics:
            feature_cols.append(f"{metric}_{i}")
    feature_cols.append("Activity")
    return pd.DataFrame(dataset_features, columns=feature_cols)

all_files = train_files + test_files
feature_df = extract_features(all_files)
feature_df.index = np.arange(1, len(feature_df) + 1)
feature_df.index.name = 'Instance'
feature_df.head()

,min_1,max_1,mean_1,median_1,std_1,1st_quart_1,3rd_quart_1,min_2,max_2,mean_2,...,1st_quart_5,3rd_quart_5,min_6,max_6,mean_6,median_6,std_6,1st_quart_6,3rd_quart_6,Activity
Instance,,,,,,,,,,,,,,,,,,,,,
1,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94,bending1
2,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00,bending1
3,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50,bending1
4,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,1.58,0.378083,...,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87,bending1
5,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,1.50,0.413125,...,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83,bending1


#### iii. Standard Deviation

In [7]:
features_only=feature_df.drop(columns=['Activity'])
std_estimates=features_only.std()
n_bootstraps=1000
ci_lower=[]
ci_upper=[]

for col in features_only.columns:
    bootstrapped_stds=[]

    for _ in range(n_bootstraps):
        sample=features_only[col].sample(frac=1, replace=True)
        bootstrapped_stds.append(sample.std())

    ci_lower.append(np.percentile(bootstrapped_stds, 5))
    ci_upper.append(np.percentile(bootstrapped_stds, 95))

bootstrap_results=pd.DataFrame({
    'Estimated_Std': std_estimates.values,
    '90%_CI_Lower': ci_lower,
    '90%_CI_Upper': ci_upper
}, index=features_only.columns)
bootstrap_results

,Estimated_Std,90%_CI_Lower,90%_CI_Upper
min_1,9.624011,8.364157,10.750075
max_1,4.207745,3.202897,5.074762
mean_1,5.276431,4.670455,5.820356
median_1,5.386624,4.735387,5.956687
std_1,1.771282,1.574637,1.953437
1st_quart_1,6.127846,5.524573,6.632873
3rd_quart_1,5.031028,4.183405,5.722181
min_2,0.000000,0.000000,0.000000
max_2,5.059656,4.636514,5.396611
mean_2,1.577908,1.394331,1.699253


#### iv. Select Features

mean establishes the baseline signal level for the activity
minimum and maximum boundaries capture the extreme ranges of motion associated with different activities

## 2. ISLR 3.7.4

### (a) Linear Train

Cubic regression will have a lower training RSS. The linear model is nested within the cubic model, the cubic model's training RSS is bounded to be less than or equal to the linear model's training RSS.

### (b) Linear Test

Linear regression will have a lower test RSS. The true underlying relationship is linear, the cubic model's extra flexibility causes it to overfit by memorizing the random noise of the training data.

### (c) Not Linear Train

Cubic regression will have a lower training RSS. It can trace both the non-linear signal of the data and the random noise much more tightly than a straight line.

### (d) Not Linear Testing

Not enough information to tell, depending on bias-variance trade-off. If true relationship is slightly non-linear then cubic regression may have higher RSS. If true relationship is highly non-linear, then cubic regression may have lower RSS.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 